#Reading From Bronze Table

# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DataType, DoubleType, IntegerType
from pyspark.sql.functions import trim, col
from pyspark.sql.functions import split, col
from pyspark.sql.window import Window

## erp_cust_az12 Transformation

In [0]:
from delta.tables import DeltaTable

In [0]:
# ============================================================
# ERP_CUST_AZ12 — Customer Birthdate & Gender
# ============================================================
df_az12 = spark.table("dev_project.bronze.erp_cust_az12")

# Trim all string columns
string_cols = [f.name for f in df_az12.schema.fields if isinstance(f.dataType, StringType)]
for col_name in string_cols:
    df_az12 = df_az12.withColumn(
        col_name,
        F.when(F.trim(F.col(col_name)) == "", None)
         .otherwise(F.trim(F.col(col_name)))
    )

# Clean CID — strip first 3 chars 'NAS' to get AW00011000 format
df_az12 = df_az12.withColumn(
    "CID",
    F.substring(F.col("CID"), 4, 100)
)

# Normalize GEN — handle M/F abbreviations and nulls
df_az12 = df_az12.withColumn(
    "GEN",
    F.when(F.upper(F.trim(F.col("GEN"))) == "M", "Male")
     .when(F.upper(F.trim(F.col("GEN"))) == "F", "Female")
     .when(F.upper(F.trim(F.col("GEN"))) == "MALE", "Male")
     .when(F.upper(F.trim(F.col("GEN"))) == "FEMALE", "Female")
     .otherwise("n/a")
)

# Null out future birthdates — cannot be born in the future
df_az12 = df_az12.withColumn(
    "BDATE",
    F.when(F.col("BDATE") > F.current_date(), None)
     .otherwise(F.col("BDATE"))
)

# Rename + final select
df_az12_silver = df_az12.select(
    F.col("CID").alias("customer_key"),
    F.col("BDATE").alias("birthdate"),
    F.col("GEN").alias("gender")
)

# Validation
print("\n" + "="*50)
print(" ERP_CUST_AZ12 VALIDATION")
print("="*50)
print(f"Total records:      {df_az12_silver.count():,}")
print(f"Null customer keys: {df_az12_silver.filter(F.col('customer_key').isNull()).count()}")
print(f"Null birthdates:    {df_az12_silver.filter(F.col('birthdate').isNull()).count()}")
print(f"Gender breakdown:")
df_az12_silver.groupBy("gender").count().show()

# Write — overwrite (reference table, current state only)
(
    df_az12_silver.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("dev_project.silver.erp_cust_az12")
)
print("erp_cust_az12 Silver write complete.")